# Pipeline z PCA + SVM vs PCA + Random Forest (Breast Cancer)

Ten notebook pokazuje kompletny, dydaktyczny przykład **Data Science pipeline**:

- stabilny zbiór danych `load_breast_cancer`
- podział train/test (ze stratyfikacją)
- dwa pipeline’y:
  1) `StandardScaler → PCA → SVM`
  2) `PCA → RandomForest`
- strojenie hiperparametrów przez `GridSearchCV`
- ewaluacja na teście: Accuracy, ROC-AUC, raport klasyfikacji
- wykresy: macierz pomyłek oraz krzywa ROC

Dlaczego to jest ciekawe? Porównujesz dwa różne „style” uczenia:
- SVM lubi dane ustandaryzowane i często korzysta z PCA.
- Random Forest nie wymaga skalowania; PCA bywa neutralne lub nawet szkodliwe, ale świetnie pokazuje wpływ transformacji na model.


In [1]:

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1) Dane: `breast_cancer`

Zbiór ma 30 cech numerycznych opisujących guz, a celem jest klasyfikacja:

- `malignant` (złośliwy)
- `benign` (łagodny)

To świetny dataset do pokazania standaryzacji, redukcji wymiaru (PCA) i różnic między SVM oraz drzewami.


In [4]:

data = load_breast_cancer()
X, y = data.data, data.target  # y: 0/1

print("Kształt X:", X.shape)
print("Cechy:", data.feature_names)
print("Rozkład klas (0=malignant, 1=benign):", np.bincount(y))
print("Nazwy klas:", data.target_names)




Kształt X: (569, 30)
Cechy: ['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness' 'mean compactness' 'mean concavity'
 'mean concave points' 'mean symmetry' 'mean fractal dimension'
 'radius error' 'texture error' 'perimeter error' 'area error'
 'smoothness error' 'compactness error' 'concavity error'
 'concave points error' 'symmetry error' 'fractal dimension error'
 'worst radius' 'worst texture' 'worst perimeter' 'worst area'
 'worst smoothness' 'worst compactness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension']
Rozkład klas (0=malignant, 1=benign): [212 357]
Nazwy klas: ['malignant' 'benign']


## 2) Podział train/test

Używamy:
- `stratify=y`, żeby zachować proporcje klas
- `random_state`, żeby wynik był powtarzalny


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape, " Test:", X_test.shape)


## 3) Pipeline 1: `StandardScaler → PCA → SVM`

**Dlaczego tak?**
- `StandardScaler` jest krytyczny dla SVM (szczególnie RBF), bo SVM jest wrażliwy na skalę cech.
- `PCA` redukuje wymiar i może zmniejszyć szum, często stabilizuje i przyspiesza SVM.
- `SVC(probability=True)` umożliwia `predict_proba`, a więc ROC-AUC i krzywą ROC.

Stroimy:
- `pca__n_components`
- jądro SVM (`linear` vs `rbf`)
- `C` oraz `gamma` (dla RBF)


In [ ]:

pipe_svm = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("pca", PCA(random_state=RANDOM_STATE)),
    ("clf", SVC(probability=True, random_state=RANDOM_STATE))
])

param_grid_svm = [
    # SVM liniowy: gamma nie ma sensu, więc osobny blok
    {
        "pca__n_components": [5, 10, 15, 20, 25, 30],
        "clf__kernel": ["linear"],
        "clf__C": [0.1, 1, 10, 100]
    },
    # SVM RBF: tu gamma ma znaczenie
    {
        "pca__n_components": [5, 10, 15, 20, 25, 30],
        "clf__kernel": ["rbf"],
        "clf__C": [0.1, 1, 10, 100],
        "clf__gamma": ["scale", 0.1, 0.01, 0.001]
    }
]

grid_svm = GridSearchCV(
    pipe_svm,
    param_grid=param_grid_svm,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)


## 4) Pipeline 2: `PCA → RandomForest`

RF nie wymaga skalowania. PCA dodajemy celowo dydaktycznie:
- pokazuje, że ta sama transformacja może pomagać jednemu modelowi, a nie pomagać drugiemu
- ułatwia porównanie na wspólnym schemacie

Stroimy:
- `pca__n_components`
- `n_estimators`, `max_depth`, `min_samples_leaf`


In [ ]:

pipe_rf = Pipeline(steps=[
    ("pca", PCA(random_state=RANDOM_STATE)),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE))
])

param_grid_rf = {
    "pca__n_components": [5, 10, 15, 20, 25, 30],
    "clf__n_estimators": [200, 500],
    "clf__max_depth": [None, 5, 10, 20],
    "clf__min_samples_leaf": [1, 2, 4],
}

grid_rf = GridSearchCV(
    pipe_rf,
    param_grid=param_grid_rf,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)


## 5) Trening i wybór najlepszych konfiguracji

`GridSearchCV`:
- sprawdza kombinacje parametrów
- robi walidację krzyżową (5-fold)
- wybiera konfigurację o najwyższym ROC-AUC


In [ ]:

grid_svm.fit(X_train, y_train)
grid_rf.fit(X_train, y_train)

print("=== SVM (best CV) ===")
print("Best params:", grid_svm.best_params_)
print("Best ROC-AUC (CV):", round(grid_svm.best_score_, 4))

print("\n=== RF (best CV) ===")
print("Best params:", grid_rf.best_params_)
print("Best ROC-AUC (CV):", round(grid_rf.best_score_, 4))


## 6) Ewaluacja na teście + wykresy

Liczymy:
- **Accuracy**: odsetek poprawnych klasyfikacji (zależny od progu)
- **ROC-AUC**: miara rankingowa (jak dobrze model rozdziela klasy w skali score’ów)

Wykresy:
- **Confusion Matrix**: TP/TN/FP/FN
- **ROC**: TPR vs FPR dla różnych progów


In [ ]:

best_svm = grid_svm.best_estimator_
best_rf = grid_rf.best_estimator_

def evaluate(name, model):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    print(f"\n--- {name} (TEST) ---")
    print("Accuracy:", round(acc, 4))
    print("ROC-AUC :", round(auc, 4))
    print("\nClassification report:\n", classification_report(
        y_test, y_pred, target_names=data.target_names
    ))

    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=data.target_names)
    plt.title(f"Confusion Matrix: {name}")
    plt.show()

    RocCurveDisplay.from_predictions(y_test, y_proba)
    plt.title(f"ROC Curve: {name}")
    plt.show()

evaluate("StandardScaler + PCA + SVM", best_svm)
evaluate("PCA + Random Forest", best_rf)


## 7) Bonus: ile wariancji wyjaśnia PCA?

To pomaga zrozumieć, czy np. 10 komponentów to dużo czy mało.
Jeśli kilka komponentów wyjaśnia większość wariancji, redukcja wymiaru ma sens.


In [ ]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

pca_full = PCA(random_state=RANDOM_STATE).fit(X_train_scaled)
cum = np.cumsum(pca_full.explained_variance_ratio_)

plt.plot(np.arange(1, len(cum)+1), cum)
plt.xlabel("Liczba komponentów PCA")
plt.ylabel("Skumulowana wyjaśniona wariancja")
plt.title("PCA: wyjaśniona wariancja vs liczba komponentów")
plt.grid(True)
plt.show()

print("Skumulowana wariancja dla 5/10/15/20 komponentów:",
      [round(cum[i-1], 4) for i in [5,10,15,20]])
